# LangChain Document Q&A

Notebook walkthrough of the PDF RAG flow using ChromaDB and the Gemini API.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()

DOCUMENTS_DIR = "documents"
CHROMA_DIR = "chroma_db"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "models/embedding-001")


In [ ]:
def read_documents(directory):
    # RAG step 1: load PDF pages from the local input folder.
    loader = PyPDFDirectoryLoader(directory)
    return loader.load()


documents = read_documents(DOCUMENTS_DIR)
len(documents)


In [ ]:
def chunk_documents(docs, chunk_size=800, chunk_overlap=100):
    # RAG step 2: split pages into overlapping chunks for precise retrieval.
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)


chunks = chunk_documents(documents)
len(chunks)


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GEMINI_API_KEY"],
)

# RAG step 3: embed chunks and persist them in ChromaDB.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)


In [ ]:
def retrieve_results(query, k=4):
    # RAG step 4: retrieve chunks closest to the question embedding.
    return vector_store.similarity_search_with_score(query, k=k)


In [ ]:
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.2,
)


def answer_question(question):
    matches = retrieve_results(question)
    context = "\n\n".join(
        f"Source: {doc.metadata.get('source')} page {doc.metadata.get('page')}\n{doc.page_content}"
        for doc, _score in matches
    )
    # RAG step 5: ask Gemini to answer using only retrieved PDF context.
    prompt = f"""
You are an AI PDF assistant. Answer the question using only the context below.
If the answer is not present in the context, say that the document does not contain enough information.

Context:
{context}

Question: {question}
Answer:
"""
    return llm.invoke(prompt).content, matches


In [ ]:
question = "What is the document about?"
answer, sources = answer_question(question)
print(answer)
